# 01 · Build the dataset subset from GenImage
**Runtime:** CPU is enough (Runtime → Change runtime type → CPU).

**Goal:** read the GenImage split-zip archives *in place* on Google Drive, list every file into a
manifest, draw a reproducible sample, extract only those files, and check them for corruption,
duplicates and train/test leakage.

**Output (on Drive, `genimage_project/`):**
- `manifests/manifest_<generator>.csv.gz`, `manifests/manifest_all.csv.gz`: every file in every archive
- `manifests/sampling_plan.csv`: which files we sample and why
- `raw_subset/<generator>/<split>/<label>/`: the sampled files, **original bytes**
- `manifests/subset_index.csv`: **the single source of truth for notebooks 02 and 03**

Never copy or extract a whole generator: each is up to ~90 GB and Colab's disk is ~80 GB.

Dataset: Zhu et al., *GenImage: A Million-Scale Benchmark for Detecting AI-Generated Image*,
NeurIPS 2023 ([arXiv 2306.08571](https://arxiv.org/abs/2306.08571)).

In [ ]:
# ===== Standard setup cell (identical in every notebook) =====
# What it does: (1) in Colab, clone the repo the first time and `git pull` afterwards,
# (2) make src/ importable, (3) run src.colab_setup.setup(): mount Drive, install deps
# (incl. apt p7zip-full), set seeds, print GPU info, create the Drive output folders.
%load_ext autoreload
%autoreload 2
import os, sys, subprocess

REPO_URL = "https://github.com/<YOUR_GITHUB_USERNAME>/genimage-detection.git"  # <-- edit once after creating the repo
REPO_DIR = "/content/genimage-detection"

if "google.colab" in sys.modules:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], capture_output=True, text=True)
        print("[setup] git pull:", (r.stdout or r.stderr).strip())
else:  # local run: the notebook lives in <repo>/notebooks
    REPO_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.colab_setup import setup
cfg, P = setup()   # cfg = dict from configs/config.yaml, P = resolved paths

## Run flags
Slow steps are protected by flags so that "Run all" can be repeated safely.
Every step is also **resumable**: finished work is detected and skipped.

In [ ]:
RUN_SPEED_TEST = True    # Step 4: extract 100 files to the LOCAL disk (cheap, nothing goes to Drive)
RUN_EXTRACTION = False   # Step 11: copy the sample to Drive. Set True only after checking step 10.
RELIST = False           # True = run `7z l` again even if a manifest file already exists

In [ ]:
import json
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from src import archive_utils as au
from src import image_utils as iu
from src import manifest as mf
from src.config import generator_names, get_logger

pd.set_option("display.width", 200, "display.max_columns", 30)
log = get_logger("01_build_dataset", P.logs)   # also writes logs/01_build_dataset.log on Drive
GEN_FOLDERS = [g["folder"] for g in cfg["generators"]]
NAMES = generator_names(cfg)
SZ = cfg["seven_zip"]
SEED = cfg["project"]["seed"]
TRAIN_GEN = cfg["train_generator"]

---
# Part A · Access and manifest
## Step 1 · 7-Zip version and the archive files on Drive
For every generator folder we list the files, count the `.zNN` parts, check that the final `.zip`
(which holds the central directory) exists, and check for gaps in the part numbering.
Archive base names are discovered here, never hardcoded.

In [ ]:
print(au.seven_zip_version(SZ["binary"]))

for gen in GEN_FOLDERS:
    gdir = P.genimage_root / gen
    if not gdir.exists():
        print(f"\n{gen}: FOLDER NOT FOUND at {gdir}")
        continue
    files = au.list_folder_files(gdir)
    print(f"\n{gen}: {len(files)} files, {files['gb'].sum():.1f} GB "
          f"(showing the first 3 and the last 2)")
    display(pd.concat([files.head(3), files.tail(2)]).drop_duplicates())

In [ ]:
inv = au.archive_inventory(P.genimage_root, GEN_FOLDERS)
inv.to_csv(P.manifests / "archive_inventory.csv", index=False)
display(inv)
bad = inv.loc[inv["status"] != "ok"]
if len(bad):
    log.warning(f"{len(bad)} generator(s) have a problem (see 'status'); they will be skipped later:\n{bad}")
print(f"Total archive size on Drive: {inv['total_gb'].sum():.0f} GB")

## Steps 2–3 · List Stable Diffusion v1.4 in place
`7z l -slt <base>.zip` reads only the central directory, so nothing is copied.
The output is parsed into a manifest with: generator, archive_path, inner_path, split, label,
ext, uncompressed_bytes and imagenet_class.

If 7-Zip cannot open the split archive, the notebook **stops here** with the exact error.
We deliberately do not try any workaround that copies the whole archive.

In [ ]:
try:
    t0 = time.time()
    m_sd, secs = mf.list_generator_to_manifest(TRAIN_GEN, inv, P.manifests, P.work_dir, cfg,
                                               relist=RELIST, logger=log)
    print(f"Listing took {time.time() - t0:.1f}s total" + (f" (7z: {secs:.1f}s)" if secs else " (loaded from disk)"))
except au.SevenZipError as e:
    raise RuntimeError(
        "STOP: 7-Zip could not open the split archive directly from Drive.\n"
        f"Exact error:\n{e}\n\n"
        "Do NOT copy the whole archive to work around this. Check that every .zNN part and the .zip "
        "are present (inventory table above) and that the Drive shortcut points at the right folder. "
        "If p7zip 16.02 cannot read this split-zip format, a newer official 7-Zip build (7zz) may: "
        "install it and set seven_zip.binary in configs/config.yaml.") from e

print(f"{len(m_sd):,} files in {TRAIN_GEN}")
display(m_sd.head())

**Verify the layout claimed by the README** (`[top]/{train,val}/{ai,nature}/file`), and check how
often an ImageNet class could be parsed from the filename:
- `synset`: a real ImageNet id such as `n01440764` was found
- `leading_int`: the filename starts with a number plus `_` (we *assume* it's a class index; look at examples)

In [ ]:
print("Top-level folders:", m_sd["top_folder"].value_counts().to_dict())
display(pd.crosstab([m_sd["split"], m_sd["label"]], m_sd["class_source"].fillna("none"), margins=True))
for (lab, src), g in m_sd.groupby(["label", m_sd["class_source"].fillna("none")]):
    print(f"{lab:<7} {src:<12} examples: {g['filename'].head(3).tolist()}")

## Step 4 · Extraction speed test
Extract 100 random files (50 ai + 50 nature, seed 42) with `7z x -i@filelist` to the **local**
disk `/content/work/test`, time it, then open each file with PIL and compare its *real* format
with its file extension.

In [ ]:
if RUN_SPEED_TEST:
    n = SZ["speed_test_n_per_label"]
    imgs = m_sd.loc[m_sd["is_image"] & m_sd["label"].isin(["ai", "nature"])]
    test = pd.concat([imgs.loc[imgs["label"] == lab].sample(n, random_state=SEED) for lab in ["ai", "nature"]])
    test_dir = P.work_dir / "test"
    shutil.rmtree(test_dir, ignore_errors=True)
    found, secs = [], 0.0
    for zp, g in test.groupby("archive_path"):
        res = au.extract_files(zp, g["inner_path"].tolist(), test_dir, SZ["binary"], SZ["extract_timeout_sec"])
        found += res["found"]
        secs += res["seconds"]
        if res["missing"]:
            log.warning(f"{len(res['missing'])} files missing after extraction, e.g. {res['missing'][:3]}")
    mb = test.loc[test["inner_path"].isin(found), "uncompressed_bytes"].sum() / 1e6
    print(f"Extracted {len(found)}/{len(test)} files, {mb:.1f} MB in {secs:.1f}s "
          f"-> {secs / max(len(found), 1):.2f} s/image, {mb / max(secs, 1e-9):.2f} MB/s")
    meta = pd.DataFrame([iu.image_metadata(test_dir / p, with_hashes=False) for p in found])
    meta["ext"] = [Path(p).suffix.lower() for p in found]
    meta["label"] = test.set_index("inner_path").loc[found, "label"].values
    print("\nReal format (from file content) vs extension:")
    display(pd.crosstab([meta["label"], meta["ext"]], meta["real_format"].fillna("UNREADABLE")))
    display(meta.groupby("label")[["width", "height", "file_bytes", "jpeg_quality"]].describe().T)
    est_h = secs / max(len(found), 1) * (2 * cfg["sampling"]["train_per_label"]
                                          + 2 * len(GEN_FOLDERS) * cfg["sampling"]["test_per_label"]) / 3600
    print(f"\nRough extrapolation for the full sample at this speed: ~{est_h:.1f} h "
          "(batches are larger than this test, so it is likely faster).")

## Step 5 · List all 8 generators
One manifest per generator in `manifests/`. Resumable (done generators are loaded, not re-listed);
an error in one generator is logged and the loop continues.

In [ ]:
manifests, status = {}, []
for gen in GEN_FOLDERS:
    try:
        t0 = time.time()
        m, secs = mf.list_generator_to_manifest(gen, inv, P.manifests, P.work_dir, cfg, relist=RELIST, logger=log)
        manifests[gen] = m
        status.append({"generator": gen, "files": len(m), "seconds": round(time.time() - t0, 1),
                       "status": "listed" if secs is not None else "loaded"})
    except Exception as e:
        log.error(f"{gen}: FAILED -> {e}")
        status.append({"generator": gen, "files": 0, "seconds": round(time.time() - t0, 1),
                       "status": f"ERROR: {str(e).splitlines()[0][:120]}"})
display(pd.DataFrame(status))

m_all = pd.concat(manifests.values(), ignore_index=True)
m_all.to_csv(P.manifests / "manifest_all.csv.gz", index=False)
print(f"manifest_all: {len(m_all):,} files from {m_all['generator'].nunique()} generators "
      f"-> {P.manifests / 'manifest_all.csv.gz'}")

## Step 6 · Manifest summary
Counts per generator × split × label, file extensions per label, and anything unexpected
(unknown split/label, non-image files, unusual top-level folders).

In [ ]:
print("Files per generator x split x label:")
display(mf.manifest_summary(m_all))

print("File extension x label (all generators):")
display(pd.crosstab(m_all["ext"], m_all["label"], margins=True))

print("Top-level folder(s) per generator:")
display(m_all.groupby("generator")["top_folder"].unique().to_frame())

unknown = m_all.loc[(m_all["split"] == "unknown") | (m_all["label"] == "unknown")]
non_img = m_all.loc[~m_all["is_image"]]
print(f"Files with unknown split/label: {len(unknown):,}")
if len(unknown):
    display(unknown.groupby(["generator", "top_folder"]).size().rename("n").reset_index().head(20))
print(f"Non-image files: {len(non_img):,}")
if len(non_img):
    display(non_img[["generator", "inner_path", "uncompressed_bytes"]].head(20))

print("ImageNet class parsing per generator x label:")
display(pd.crosstab([m_all["generator"], m_all["label"]], m_all["class_source"].fillna("none")))

## Step 7 · Leakage check by filename
The real ("nature") images all come from ImageNet. If the same real filename appears in several
generators, or in both train and val, a real image could end up in both our training and test sets.

In [ ]:
leak = mf.filename_leakage_report(m_all)
print(json.dumps(leak["summary"], indent=2))
print("\nNumber of shared 'nature' filenames between generators (diagonal = total per generator):")
display(leak["overlap_matrix"])
json.dump(leak["summary"], open(P.results / "filename_leakage_summary.json", "w"), indent=2)

---
# Part B · Sampling and extraction
## Steps 8–9 · Draw the sample
- **Train pool:** SD1.4 `train` split: 5000 ai + 5000 nature.
- **Test sets:** each generator's `val` split: 500 ai + 500 nature (SD1.4 val is the in-distribution test set).
- Seed 42, stratified by ImageNet class where a class could be parsed (round-robin over classes).
- Step 9: any nature image whose filename is in the training sample is removed from the test
  pools *before* sampling, so test sets keep their full size.

In [ ]:
plan = mf.build_sampling_plan(m_all, cfg, logger=log)
plan.to_csv(P.manifests / "sampling_plan.csv", index=False)
display(plan.groupby(["generator", "role", "label"]).size().unstack("label"))

overlap = set(plan.loc[plan.role == "train", "filename"]) & set(plan.loc[plan.role == "test", "filename"])
print(f"Filenames shared by train and test samples: {len(overlap)} (must be 0)")
assert not overlap

print("ImageNet class coverage of the sample:")
display(mf.class_coverage(plan))

## Step 10 · Size estimate before extracting
From `uncompressed_bytes` in the manifest. If the sample is larger than the Drive budget
(`sampling.drive_budget_gb` in the config), the notebook stops and suggests smaller sample sizes.

In [ ]:
size = plan.groupby(["generator", "role", "label"])["uncompressed_bytes"].sum().div(1e9).round(3)
display(size.unstack("label"))
total_gb = mf.estimate_size_gb(plan)
budget = cfg["sampling"]["drive_budget_gb"]
print(f"Estimated raw subset size: {total_gb:.2f} GB (budget {budget} GB)")
if total_gb > budget:
    sug = mf.suggest_sample_sizes(plan, budget, cfg["sampling"]["train_per_label"], cfg["sampling"]["test_per_label"])
    raise RuntimeError(f"STOP: sample is over the Drive budget. Suggestion: {sug}. "
                       "Edit sampling.* in configs/config.yaml, push, and re-run.")
print("OK: within budget.")

## Step 11 · Extract the sample and copy it to Drive
For each generator: `7z x -i@list` (in batches) to the local disk → compute metadata on the fast
local copy → copy the **original bytes** to `raw_subset/<generator>/<split>/<label>/`.
Resumable: files already on Drive with the right size are skipped.
Per-file metadata (format, size, JPEG quality, md5, pHash) is saved to `manifests/raw_metadata_<generator>.csv`.

In [ ]:
def on_file(local_path, row):
    """Metadata computed on the local copy, before it is deleted."""
    return iu.image_metadata(local_path, with_hashes=True, hash_size=cfg["dedup"]["phash_size"])


if not RUN_EXTRACTION:
    print("RUN_EXTRACTION = False -> skipping extraction. Set it to True at the top to run this step.")
else:
    summary = []
    for gen in GEN_FOLDERS:
        sub = plan.loc[plan["generator"] == gen]
        if sub.empty:
            log.warning(f"{gen}: nothing sampled (listing failed?) -> skipped")
            continue
        t0 = time.time()
        log.info(f"=== {gen}: {len(sub)} files, {sub['uncompressed_bytes'].sum() / 1e9:.2f} GB ===")
        try:
            for zp, s in sub.groupby("archive_path"):
                st, extra = au.extract_sample_to_drive(
                    s, zp, P.work_dir / "extract" / gen, P.project_root, batch_size=SZ["extract_batch_size"],
                    binary=SZ["binary"], timeout=SZ["extract_timeout_sec"], on_file=on_file, logger=log)
                st.to_csv(P.logs / f"extraction_status_{gen}.csv", mode="a", index=False,
                          header=not (P.logs / f"extraction_status_{gen}.csv").exists())
                if extra:
                    mfile = P.manifests / f"raw_metadata_{gen}.csv"
                    pd.DataFrame(extra).to_csv(mfile, mode="a", index=False, header=not mfile.exists())
                counts = st["status"].value_counts().to_dict()
                summary.append({"generator": gen, **counts, "minutes": round((time.time() - t0) / 60, 1)})
                log.info(f"{gen}: {counts}")
        except Exception as e:
            log.error(f"{gen}: extraction FAILED -> {e}")
            summary.append({"generator": gen, "error": str(e).splitlines()[0][:150]})
    display(pd.DataFrame(summary))

## Step 12 · Verify, hash and remove leakage
1. Every sampled file must exist on Drive and open with PIL.
2. Metadata missing from step 11 (e.g. after a disconnect) is computed now, from Drive.
3. **Exact duplicates** (same md5) and **near duplicates** (pHash Hamming distance ≤ `dedup.near_dup_hamming`)
   within and across generators, and between train and test.
4. Test images that duplicate a training image are dropped and logged.

In [ ]:
plan = mf.read_csv(P.manifests / "sampling_plan.csv")
plan["exists"] = [(P.project_root / r).exists() for r in plan["raw_relpath"]]
print(f"Sampled files present on Drive: {plan['exists'].sum():,}/{len(plan):,}")
if not plan["exists"].all():
    display(plan.loc[~plan["exists"]].groupby(["generator", "role", "label"]).size().rename("missing"))

meta_files = sorted(P.manifests.glob("raw_metadata_*.csv"))
meta = pd.concat([mf.read_csv(f) for f in meta_files], ignore_index=True) if meta_files else pd.DataFrame(columns=["sample_id"])
if "on_file_error" in meta:
    meta = meta.loc[meta["on_file_error"].isna()].drop(columns="on_file_error")
meta = meta.drop_duplicates("sample_id", keep="last")
need = plan.loc[plan["exists"] & ~plan["sample_id"].isin(meta["sample_id"])]
print(f"Metadata available for {len(meta):,} files; computing it for {len(need):,} more from Drive...")
if len(need):
    extra = iu.metadata_many([P.project_root / r for r in need["raw_relpath"]], n_threads=cfg["dedup"]["n_threads"],
                             hash_size=cfg["dedup"]["phash_size"])
    extra = pd.DataFrame(extra).assign(sample_id=need["sample_id"].values)
    meta = pd.concat([meta, extra], ignore_index=True)
meta.to_csv(P.manifests / "raw_metadata.csv", index=False)

sub = plan.loc[plan["exists"]].merge(meta, on="sample_id", how="left")
unreadable = sub.loc[sub["readable"] != True]
print(f"Unreadable / corrupt files: {len(unreadable)}")
if len(unreadable):
    display(unreadable[["sample_id", "error"]].head(20))

In [ ]:
ok = sub.loc[sub["readable"] == True].reset_index(drop=True)
pairs = mf.duplicate_pairs(ok, cfg["dedup"]["near_dup_hamming"])
pairs.to_csv(P.manifests / "duplicate_pairs.csv", index=False)
print(f"{len(pairs)} duplicate pairs (exact = same bytes; near = pHash distance <= {cfg['dedup']['near_dup_hamming']})")
if len(pairs):
    display(pairs.groupby(["kind", "roles", "same_generator"]).size().rename("pairs").reset_index())
    display(pairs.groupby(["kind", "a_label", "b_label"]).size().rename("pairs").reset_index())

Look at a few near-duplicate pairs to judge whether the threshold is sensible
(true duplicates should look identical; if they don't, lower `dedup.near_dup_hamming`).

In [ ]:
from src.eda_utils import plot_image_grid, savefig, set_style
set_style()
show = pairs.sort_values("hamming", ascending=False).head(6)
if len(show):
    rel = ok.set_index("sample_id")["raw_relpath"]
    paths, caps = [], []
    for _, r in show.iterrows():
        for side in "ab":
            paths.append(P.project_root / rel[r[f"{side}_sample_id"]])
            caps.append(f"{r[f'{side}_generator'][:12]}/{r[f'{side}_role']}/{r[f'{side}_label']}\n{r['kind']} d={r['hamming']}")
    fig = plot_image_grid(paths, caps, ncols=4, title="Duplicate pairs (side by side)")
    savefig(fig, "01_duplicate_pairs", P.figures)

In [ ]:
leaks = mf.leaking_test_ids(pairs) if len(pairs) else set()
dropped = pd.concat([
    sub.loc[sub["sample_id"].isin(leaks)].assign(drop_reason="duplicate of a training image"),
    unreadable.assign(drop_reason="unreadable"),
    plan.loc[~plan["exists"]].assign(drop_reason="missing on Drive"),
], ignore_index=True)
dropped.to_csv(P.manifests / "dropped_files.csv", index=False)
log.info(f"Dropped {len(dropped)} files: {dropped['drop_reason'].value_counts().to_dict()}")
display(dropped.groupby(["drop_reason", "generator", "role", "label"]).size().rename("n").reset_index())

## Step 13 · Save `subset_index.csv`
One row per kept image: path (relative to `genimage_project/`), generator, archive split,
role (train/test), label, y (1 = fake), class, and all metadata. **Later notebooks read only this file.**

In [ ]:
keep = sub.loc[(sub["readable"] == True) & ~sub["sample_id"].isin(leaks)].copy()
keep["generator_name"] = keep["generator"].map(NAMES)
cols = ["sample_id", "raw_relpath", "generator", "generator_name", "split", "role", "label", "y",
        "filename", "ext", "imagenet_class", "class_source", "uncompressed_bytes",
        "readable", "real_format", "width", "height", "mode", "file_bytes", "jpeg_quality", "jpeg_q_exact",
        "is_gray", "md5", "phash", "inner_path", "archive_path"]
subset_index = keep[cols].sort_values(["generator", "role", "label", "filename"]).reset_index(drop=True)
subset_index.to_csv(P.manifests / "subset_index.csv", index=False)
print(f"Saved {P.manifests / 'subset_index.csv'}: {len(subset_index):,} images")
display(subset_index.groupby(["generator", "role", "label"]).size().unstack("label"))
du = sum(f.stat().st_size for f in P.raw_subset.rglob("*") if f.is_file()) / 1e9
print(f"raw_subset/ uses {du:.2f} GB on Drive")

---
## Findings (fill in after running)
- Archive access: 7z version, did the split archives open in place? Listing time per generator:
- Actual inner layout vs README (top folder, split/label folders, unexpected files):
- Extraction speed (s/image, MB/s) and total extraction time:
- Extensions / real formats of ai vs nature files:
- ImageNet class parsing (which pattern, coverage per generator):
- Filename leakage (shared nature filenames across generators / train-val):
- Duplicates found and files dropped:
- Final subset size (images, GB):